# INDI BLOB Test
Interaktives Debugging der BLOB-Übertragung von der Canon DSLR.

In [ ]:
import time
import PyIndi

INDI_HOST = '192.168.2.12'
INDI_PORT = 7624

In [ ]:
class TestClient(PyIndi.BaseClient):
    def __init__(self):
        super().__init__()
        self.devices = {}
        self.blob_data = None
    def newDevice(self, d):
        self.devices[d.getDeviceName()] = d
        print(f'Device: {d.getDeviceName()}')
    def newProperty(self, p): pass
    def removeProperty(self, p): pass
    def newBLOB(self, bp):
        for i in range(bp.count()):
            self.blob_data = bp[i].getblobdata()
            print(f'*** BLOB received: {len(self.blob_data)} bytes, format={bp[i].getFormat()}')
    def newSwitch(self, s): pass
    def newNumber(self, n): pass
    def newText(self, t): pass
    def newLight(self, l): pass
    def newMessage(self, d, m):
        try:
            print(f'MSG [{d.getDeviceName()}]: {d.messageQueue(m)}')
        except: pass
    def serverConnected(self): print('Connected!')
    def serverDisconnected(self, c): print(f'Disconnected: {c}')

client = TestClient()
client.setServer(INDI_HOST, INDI_PORT)
client.connectServer()
time.sleep(3)
print(f'Devices: {list(client.devices.keys())}')

In [ ]:
# Find camera
camera = None
for d in client.devices.values():
    if d.getNumber('CCD_EXPOSURE'):
        camera = d
        print(f'Camera: {d.getDeviceName()}')
        break

if not camera:
    print('No camera found!')

In [ ]:
# List all BLOB-related properties
if camera:
    for p in camera.getProperties():
        name = p.getName()
        ptype = p.getTypeAsString()
        if any(x in name.lower() for x in ['blob', 'upload', 'transfer', 'format', 'ccd']):
            print(f'{ptype:8s} {name}')
            if p.getType() == PyIndi.INDI_SWITCH:
                sw = p.getSwitch()
                for i in range(sw.count()):
                    print(f'           {sw[i].getName()} = {sw[i].getState()}')
            elif p.getType() == PyIndi.INDI_NUMBER:
                num = p.getNumber()
                for i in range(num.count()):
                    print(f'           {num[i].getName()} = {num[i].value}')

In [ ]:
# Configure BLOB mode + upload
if camera:
    client.enableDirectBlobAccess(camera.getDeviceName(), None)
    client.setBLOBMode(PyIndi.B_ALSO, camera.getDeviceName(), None)
    
    upload = camera.getSwitch('UPLOAD_MODE')
    if upload:
        for i in range(upload.count()):
            upload[i].setState(PyIndi.ISS_ON if upload[i].getName() == 'UPLOAD_CLIENT' else PyIndi.ISS_OFF)
        client.sendNewSwitch(upload)
        print('Upload mode: CLIENT')
    
    print('BLOB mode configured')

In [ ]:
# Take a 1s exposure and wait for BLOB
if camera:
    client.blob_data = None
    exp = camera.getNumber('CCD_EXPOSURE')
    exp[0].value = 1.0
    client.sendNewNumber(exp)
    print('Exposure started: 1s')
    
    for i in range(60):
        time.sleep(1)
        if client.blob_data:
            print(f'Got BLOB at t={i}s: {len(client.blob_data)} bytes')
            break
    else:
        print('No BLOB after 60s')

In [ ]:
# Disconnect
client.disconnectServer()
print('Disconnected')